# Org shot classification

In [ ]:
import math
import json
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd

# ============================================================================
# Helper Functions
# ============================================================================

def parse_xy(val) -> Optional[Tuple[float, float]]:
    """Parse [x,y] stored as list/tuple or JSON-like string; None if not parseable."""
    if isinstance(val, (list, tuple)) and len(val) >= 2:
        try:
            return float(val[0]), float(val[1])
        except Exception:
            return None
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return None
        try:
            j = json.loads(s.replace("'", '"'))
            if isinstance(j, (list, tuple)) and len(j) >= 2:
                return float(j[0]), float(j[1])
        except Exception:
            return None
    return None

def dist_to_nearest_goal(xy: Tuple[float, float]) -> float:
    """Euclidean distance from (x,y) to the nearest goal center in SB units (≈ yards)."""
    x, y = xy
    d_left  = math.hypot(x - 0.0,   y - 40.0)
    d_right = math.hypot(x - 120.0, y - 40.0)
    return min(d_left, d_right)

def get_pass_row_by_id(events: pd.DataFrame, pass_id: Any) -> Optional[pd.Series]:
    """Find the pass row by its `id` (string match after strip)."""
    if "id" not in events.columns:
        return None
    m = events[events["id"].astype(str).str.strip() == str(pass_id).strip()]
    return None if m.empty else m.iloc[0]

# ============================================================================
# Extract Important Moments
# ============================================================================

def important_moments_from_file(
    filename: str,
    xg_threshold: float = 0.08,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    """
    Returns (moments_df, moments_list) for a match events CSV.
    Each moment includes the original event `id`.

    Moments:
      - Shots with xG >= xg_threshold OR shot_outcome == "Goal"
      - Own Goal Against
      - Foul Won with foul_won_penalty == True  (labeled as 'Penalty Won')

    Post-processing:
      - Filter out events after 120:00 if minute/second exist
      - Sort by period, minute, second, timestamp (when present)
      - If `possession` exists, keep the last event per possession (dedupe)
      - Make values JSON-safe (NaN/inf -> None; numpy scalars -> native)
    """
    df = pd.read_csv(filename)

    # --- Prepare shots & xG ---
    shots = df[df["type"] == "Shot"].copy() if "type" in df.columns else pd.DataFrame()
    if not shots.empty:
        if "shot_statsbomb_xg" in shots.columns:
            shots["xg"] = pd.to_numeric(shots["shot_statsbomb_xg"], errors="coerce")
        else:
            shots["xg"] = np.nan
        shot_mask = (shots["xg"] >= xg_threshold) | (shots.get("shot_outcome", "") == "Goal")
        shots = shots[shot_mask]
    # --- Own goals ---
    own_goals = df[df["type"] == "Own Goal Against"].copy() if "type" in df.columns else pd.DataFrame()
    if not own_goals.empty:
        own_goals["xg"] = np.nan  # no xG for own goals

    # --- Penalty won (foul won with penalty flag) ---
    pen_won = pd.DataFrame()
    if "type" in df.columns and "foul_won_penalty" in df.columns:
        pen_won = df[(df["type"] == "Foul Won") & (df["foul_won_penalty"] == True)].copy()
        if not pen_won.empty:
            pen_won["xg"] = np.nan  # not a shot, but keep xg column for schema consistency

    # --- Combine candidates ---
    moments = pd.concat([shots, own_goals, pen_won], ignore_index=True)

    # Nothing matched
    if moments.empty:
        return moments, []

    # --- Time filter: remove events strictly after 120:00 ---
    if {"minute", "second"}.issubset(moments.columns):
        mask = (moments["minute"] < 120) | ((moments["minute"] == 120) & (moments["second"].fillna(0) <= 0))
        moments = moments[mask].copy()

    # --- Sort by match order ---
    sort_cols = [c for c in ["period", "minute", "second", "timestamp"] if c in moments.columns]
    if sort_cols:
        moments = moments.sort_values(sort_cols, ascending=True, kind="stable")

    # --- Group by possession: keep last in each possession (if possession exists) ---
    if "possession" in moments.columns:
        # Secondary sort to ensure "last()" is deterministic with timestamp when available
        sort_cols2 = [c for c in ["possession", "period", "minute", "second", "timestamp"] if c in moments.columns]
        moments = moments.sort_values(sort_cols2, ascending=True, kind="stable")
        moments = moments.groupby("possession", as_index=False).last()

    # --- Drop exact duplicates if any ---
    moments = moments.drop_duplicates()

    # --- Create a tidy projection with stable columns ---
    def safe(val):
        if val is None:
            return None
        if isinstance(val, float) and (math.isnan(val) or math.isinf(val)):
            return None
        return val

    # Build a clean DataFrame with key fields; keep event `id`
    keep_cols = []
    for c in ["id","match_id","type","team","player","minute","second","timestamp","period","possession","xg","shot_outcome"]:
        if c in moments.columns:
            keep_cols.append(c)

    moments_df = moments[keep_cols].copy()

    # Label "Penalty Won" outcome for the Foul Won + flag case (nice-to-have)
    if {"type", "shot_outcome", "foul_won_penalty"}.issubset(moments.columns):
        mask_pw = (moments["type"] == "Foul Won") & (moments["foul_won_penalty"] == True)
        moments_df.loc[mask_pw.values, "shot_outcome"] = "Penalty Won"

    # JSON-safe list of dicts
    def json_fallback(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        return str(o)

    # Replace NaNs with None for JSON
    moments_df = moments_df.replace([np.nan, np.inf, -np.inf], None)

    moments_list: List[Dict[str, Any]] = []
    for _, row in moments_df.iterrows():
        # Compact "who" label like "C. Immobile"
        who = "N/A"
        player = row.get("player", None)
        if isinstance(player, str) and player.strip():
            parts = player.split()
            who = f"{parts[0][0]}. {parts[-1]}" if len(parts) >= 2 else parts[0]

        # "when" as mm:ss if minute/second present
        m, s = row.get("minute", None), row.get("second", None)
        when = f"{int(m)}:{int(s):02d}" if (m is not None and s is not None) else row.get("timestamp", None) or "N/A"

        item = {
            "event_id": row.get("id"),                           # <-- include original event id
            "match_id": row.get("match_id", None),
            "possession": row.get("possession", None),
            "who": who,
            "when": when,
            "event_type": row.get("type", None),
            "xg": row.get("xg", None),
            "outcome": row.get("shot_outcome", None),
            "team": row.get("team", None),
        }
        # JSON-safe coercion
        for k, v in list(item.items()):
            try:
                item[k] = v if v is None else json.loads(json.dumps(v, default=json_fallback))
            except Exception:
                item[k] = None if v is None else str(v)
        moments_list.append(item)

    return moments_df, moments_list

# ============================================================================
# Classify Shot Moments
# ============================================================================

def classify_shot_moments(
    filename: str,
    moments_df: pd.DataFrame,
    long_shot_yards: float = 22.0
) -> pd.DataFrame:
    """
    Classify important moments where type == 'Shot' with multi-labels and xG.

    Labels:
      - From key pass flags (on the pass row referenced by `shot_key_pass_id`):
          pass_through_ball, pass_cross, pass_cut_back
      - 'Long Shot' if distance to nearest goal center >= long_shot_yards
      - 'other' if none matched

    Returns a DataFrame with one row per shot moment.
    """
    # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
    ev = pd.read_csv(filename)

    # Tolerant type column
    type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
    if type_col is None:
        raise ValueError("No 'type'/'Type' column in events CSV.")

    # Shots in the full events (we'll use this to get fields not present in moments_df)
    ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

    # Ensure an xg column exists on ev_shots
    if "shot_statsbomb_xg" in ev_shots.columns:
        ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
    else:
        ev_shots["xg"] = np.nan

    # Focus only on the events that are marked as important moments AND are shots
    if "type" not in moments_df.columns:
        raise ValueError("moments_df must include a 'type' column.")
    shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

    # If moments_df didn't carry 'id', fall back to intersect by time/team (not ideal). We require 'id'.
    if not shot_ids:
        raise ValueError("moments_df must include 'id' for shot events to classify them.")

    shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

    # Prepare output rows
    out_rows: List[Dict[str, Any]] = []

    for _, r in shots_focus.iterrows():
        labels: List[str] = []

        # 1) pass-based labels via key pass flags
        kp = r.get("shot_key_pass_id", np.nan)
        if pd.notna(kp):
            prow = get_pass_row_by_id(ev, kp)
            if prow is not None:
                # Only the three you asked for
                if "pass_through_ball" in prow.index and pd.notna(prow["pass_through_ball"]) and bool(prow["pass_through_ball"]):
                    labels.append("pass_through_ball")
                if "pass_cross" in prow.index and pd.notna(prow["pass_cross"]) and bool(prow["pass_cross"]):
                    labels.append("pass_cross")
                if "pass_cut_back" in prow.index and pd.notna(prow["pass_cut_back"]) and bool(prow["pass_cut_back"]):
                    labels.append("pass_cut_back")

        # 2) long shot (>= threshold yards to nearest goal center)
        loc = parse_xy(r.get("location"))
        dist_goal = float("nan")
        if loc is not None:
            dist_goal = dist_to_nearest_goal(loc)
            if dist_goal >= float(long_shot_yards):
                labels.append("Long Shot")

        # 3) other if nothing else matched
        if len(labels) == 0:
            labels.append("other")

        out_rows.append({
            "id": r.get("id"),
            "match_id": r.get("match_id"),
            "team": r.get("team"),
            "player": r.get("player"),
            "minute": r.get("minute"),
            "second": r.get("second"),
            "shot_outcome": r.get("shot_outcome"),
            "xg": r.get("xg"),
            "distance_to_goal": dist_goal,
            "labels": labels,
            "labels_str": ";".join(labels),
            "shot_key_pass_id": r.get("shot_key_pass_id"),
        })

    shot_moments_labeled = pd.DataFrame(out_rows)

    # Keep only the original important shot ids order if you want
    order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
    if "id" in shot_moments_labeled.columns:
        shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
        shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

    return shot_moments_labeled

# ============================================================================
# Usage Example
# ============================================================================

if __name__ == "__main__":
    # Set your file path
    FILENAME = "../events_by_match/events_3788741.csv"
    # Step 1: Extract important moments
    print("Extracting important moments...")
    moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)
    
    print(f"\nFound {len(moments_df)} important moments")
    print("\nMoments DataFrame:")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(moments_df.to_string(index=False))
    
    # Step 2: Classify shots from important moments
    print("\n\nClassifying shots...")
    shot_moments_labeled = classify_shot_moments(FILENAME, moments_df, long_shot_yards=22.0)
    
    print(f"\nClassified {len(shot_moments_labeled)} shots")
    print("\nLabeled Shots DataFrame:")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(shot_moments_labeled.to_string(index=False))
    
    # Optional: Save results
    # moments_df.to_csv("important_moments.csv", index=False)
    # shot_moments_labeled.to_csv("shot_moments_labeled.csv", index=False)
    # with open("important_moments.json", "w", encoding="utf-8") as f:
    #     json.dump(moments_list, f, ensure_ascii=False, indent=2)

Extracting important moments...

Found 7 important moments

Moments DataFrame:
                                  id  match_id             type   team           player  minute  second    timestamp  period  possession        xg shot_outcome
74750b41-c816-4a6c-88d3-b98859039c74   3788741             Shot  Italy  Lorenzo Insigne      17       3 00:17:03.251       1          38  0.098552        Off T
c07133d1-fa88-45f8-8a40-7be381ddffd1   3788741             Shot  Italy Domenico Berardi      44      33 00:44:33.641       1         108  0.128881        Off T
4a9f326b-e5c1-4d81-bc98-d0992cf85a00   3788741 Own Goal Against Turkey    Merih Demiral      52      12 00:07:12.643       2         125      None         None
c12c958c-b795-4471-b81c-c16582d8205a   3788741             Shot  Italy  Lorenzo Insigne      56      21 00:11:21.502       2         133  0.086273      Blocked
506e2e45-cc7a-4aa8-9c0d-2c8fb339bc2f   3788741             Shot  Italy    Ciro Immobile      65      33 00:20:33.192     

# Case 1: pass_cross/through ball

In [16]:
# %% Imports
import math
import json
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd

In [17]:
# %% Core Helper Functions
def parse_xy(val) -> Optional[Tuple[float, float]]:
    """Parse [x,y] stored as list/tuple or JSON-like string; None if not parseable."""
    if isinstance(val, (list, tuple)) and len(val) >= 2:
        try:
            return float(val[0]), float(val[1])
        except Exception:
            return None
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return None
        try:
            j = json.loads(s.replace("'", '"'))
            if isinstance(j, (list, tuple)) and len(j) >= 2:
                return float(j[0]), float(j[1])
        except Exception:
            return None
    return None

def dist_to_nearest_goal(xy: Tuple[float, float]) -> float:
    """Euclidean distance from (x,y) to the nearest goal center in SB units (≈ yards).
    
    Goals are fixed at:
    - Left goal: (0, 40)
    - Right goal: (120, 40)
    """
    x, y = xy
    d_left  = math.hypot(x - 0.0,   y - 40.0)
    d_right = math.hypot(x - 120.0, y - 40.0)
    return min(d_left, d_right)

def get_pass_row_by_id(events: pd.DataFrame, pass_id: Any) -> Optional[pd.Series]:
    """Find the pass row by its `id` (string match after strip)."""
    if "id" not in events.columns:
        return None
    m = events[events["id"].astype(str).str.strip() == str(pass_id).strip()]
    return None if m.empty else m.iloc[0]

In [18]:
# %% Calculate distance pass travels toward goal
def pass_distance_toward_goal(pass_location: Tuple[float, float], 
                                pass_end_location: Tuple[float, float]) -> float:
    """
    Calculate how far a pass travels toward the goal.
    
    The goal is fixed at (120, 40) - the right side of the pitch.
    We calculate distance to this goal from both locations and see if the pass
    moves closer to the goal.
    
    Args:
        pass_location: Starting location (x, y)
        pass_end_location: Ending location (x, y)
    
    Returns:
        Distance traveled toward goal (positive if moving toward, 0 if not)
    """
    x1, y1 = pass_location
    x2, y2 = pass_end_location
    
    # Distance to goal (120, 40) from start location
    dist_start_to_goal = math.hypot(x1 - 120.0, y1 - 40.0)
    
    # Distance to goal (120, 40) from end location
    dist_end_to_goal = math.hypot(x2 - 120.0, y2 - 40.0)
    
    # Distance traveled toward goal (positive if end is closer than start)
    distance_toward_goal = dist_start_to_goal - dist_end_to_goal
    
    # Return positive value only if moving toward goal, otherwise 0
    return max(0.0, distance_toward_goal)

In [19]:
# %% Check Case 1: pass_cross = 0 AND pass travels > 5m toward goal
def check_case1_pass_toward_goal(events_df: pd.DataFrame, 
                                  pass_id: Any, 
                                  threshold_meters: float = 5.0) -> bool:
    """
    Check if a pass (referenced by pass_id) satisfies Case 1:
    - pass_cross column = 0 (False/0/NaN)
    - Pass travels > threshold_meters toward goal
    
    Args:
        events_df: Full events DataFrame
        pass_id: ID of the pass event to check
        threshold_meters: Distance threshold (default 5.0 meters)
    
    Returns:
        True if both conditions are met, False otherwise
    """
    # Find the pass row
    pass_row = get_pass_row_by_id(events_df, pass_id)
    if pass_row is None:
        return False
    
    # Check pass_cross = 0 (False, 0, or NaN)
    pass_cross_val = pass_row.get("pass_cross", np.nan)
    is_not_cross = (
        pd.isna(pass_cross_val) or 
        pass_cross_val == False or 
        pass_cross_val == 0 or
        (isinstance(pass_cross_val, str) and pass_cross_val.lower() in ['false', '0', ''])
    )
    
    if not is_not_cross:
        return False  # pass_cross is not 0, so Case 1 fails
    
    # Get pass locations
    pass_loc = parse_xy(pass_row.get("location"))
    pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
    
    if pass_loc is None or pass_end_loc is None:
        return False  # Missing location data
    
    # Calculate distance traveled toward goal
    dist_toward_goal = pass_distance_toward_goal(pass_loc, pass_end_loc)
    
    # Check if distance > threshold
    return dist_toward_goal > threshold_meters

In [20]:
# %% Classify shot moments with Case 1 (exclusive label)
def classify_shot_moments(
    filename: str,
    moments_df: pd.DataFrame,
    long_shot_yards: float = 22.0,
    case1_threshold_meters: float = 5.0
) -> pd.DataFrame:
    """
    Classify important moments where type == 'Shot' with multi-labels and xG.
    
    IMPORTANT: Case 1 is EXCLUSIVE - if it applies, label is ONLY "case1_pass_through_ball"
    
    Labels:
      - Case 1 (exclusive): 'case1_pass_through_ball' if pass_cross=0 AND pass travels > threshold toward goal
      - If Case 1 doesn't apply, then check:
          - pass_through_ball, pass_cross, pass_cut_back (from key pass flags)
          - 'Long Shot' if distance to nearest goal center >= long_shot_yards
          - 'other' if none matched

    Returns a DataFrame with one row per shot moment.
    """
    # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
    ev = pd.read_csv(filename)

    # Tolerant type column
    type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
    if type_col is None:
        raise ValueError("No 'type'/'Type' column in events CSV.")

    # Shots in the full events
    ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

    # Ensure an xg column exists on ev_shots
    if "shot_statsbomb_xg" in ev_shots.columns:
        ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
    else:
        ev_shots["xg"] = np.nan

    # Focus only on the events that are marked as important moments AND are shots
    if "type" not in moments_df.columns:
        raise ValueError("moments_df must include a 'type' column.")
    shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

    if not shot_ids:
        raise ValueError("moments_df must include 'id' for shot events to classify them.")

    shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

    # Prepare output rows
    out_rows: List[Dict[str, Any]] = []

    for _, r in shots_focus.iterrows():
        labels: List[str] = []
        case1_applies = False

        # Check if shot has a key pass
        kp = r.get("shot_key_pass_id", np.nan)
        if pd.notna(kp):
            # FIRST: Check Case 1 (exclusive)
            if check_case1_pass_toward_goal(ev, kp, threshold_meters=case1_threshold_meters):
                labels.append("case1_pass_through_ball")
                case1_applies = True
            else:
                # Only check other labels if Case 1 doesn't apply
                prow = get_pass_row_by_id(ev, kp)
                if prow is not None:
                    # Original pass flags
                    if "pass_through_ball" in prow.index and pd.notna(prow["pass_through_ball"]) and bool(prow["pass_through_ball"]):
                        labels.append("pass_through_ball")
                    if "pass_cross" in prow.index and pd.notna(prow["pass_cross"]) and bool(prow["pass_cross"]):
                        labels.append("pass_cross")
                    if "pass_cut_back" in prow.index and pd.notna(prow["pass_cut_back"]) and bool(prow["pass_cut_back"]):
                        labels.append("pass_cut_back")

        # Long shot check (only if Case 1 doesn't apply)
        if not case1_applies:
            loc = parse_xy(r.get("location"))
            dist_goal = float("nan")
            if loc is not None:
                dist_goal = dist_to_nearest_goal(loc)
                if dist_goal >= float(long_shot_yards):
                    labels.append("Long Shot")

        # Other label (only if Case 1 doesn't apply and no other labels)
        if not case1_applies and len(labels) == 0:
            labels.append("other")

        # Calculate distance to goal for all shots (for reference)
        loc = parse_xy(r.get("location"))
        dist_goal = float("nan")
        if loc is not None:
            dist_goal = dist_to_nearest_goal(loc)

        out_rows.append({
            "id": r.get("id"),
            "match_id": r.get("match_id"),
            "team": r.get("team"),
            "player": r.get("player"),
            "minute": r.get("minute"),
            "second": r.get("second"),
            "shot_outcome": r.get("shot_outcome"),
            "xg": r.get("xg"),
            "distance_to_goal": dist_goal,
            "labels": labels,
            "labels_str": ";".join(labels),
            "shot_key_pass_id": r.get("shot_key_pass_id"),
        })

    shot_moments_labeled = pd.DataFrame(out_rows)

    # Keep only the original important shot ids order if you want
    order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
    if "id" in shot_moments_labeled.columns:
        shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
        shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

    return shot_moments_labeled

In [21]:
# %% Extract Important Moments
def important_moments_from_file(
    filename: str,
    xg_threshold: float = 0.08,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    """
    Returns (moments_df, moments_list) for a match events CSV.
    Each moment includes the original event `id`.

    Moments:
      - Shots with xG >= xg_threshold OR shot_outcome == "Goal"
      - Own Goal Against
      - Foul Won with foul_won_penalty == True  (labeled as 'Penalty Won')

    Post-processing:
      - Filter out events after 120:00 if minute/second exist
      - Sort by period, minute, second, timestamp (when present)
      - If `possession` exists, keep the last event per possession (dedupe)
      - Make values JSON-safe (NaN/inf -> None; numpy scalars -> native)
    """
    df = pd.read_csv(filename)

    # --- Prepare shots & xG ---
    shots = df[df["type"] == "Shot"].copy() if "type" in df.columns else pd.DataFrame()
    if not shots.empty:
        if "shot_statsbomb_xg" in shots.columns:
            shots["xg"] = pd.to_numeric(shots["shot_statsbomb_xg"], errors="coerce")
        else:
            shots["xg"] = np.nan
        shot_mask = (shots["xg"] >= xg_threshold) | (shots.get("shot_outcome", "") == "Goal")
        shots = shots[shot_mask]
    # --- Own goals ---
    own_goals = df[df["type"] == "Own Goal Against"].copy() if "type" in df.columns else pd.DataFrame()
    if not own_goals.empty:
        own_goals["xg"] = np.nan  # no xG for own goals

    # --- Penalty won (foul won with penalty flag) ---
    pen_won = pd.DataFrame()
    if "type" in df.columns and "foul_won_penalty" in df.columns:
        pen_won = df[(df["type"] == "Foul Won") & (df["foul_won_penalty"] == True)].copy()
        if not pen_won.empty:
            pen_won["xg"] = np.nan  # not a shot, but keep xg column for schema consistency

    # --- Combine candidates ---
    moments = pd.concat([shots, own_goals, pen_won], ignore_index=True)

    # Nothing matched
    if moments.empty:
        return moments, []

    # --- Time filter: remove events strictly after 120:00 ---
    if {"minute", "second"}.issubset(moments.columns):
        mask = (moments["minute"] < 120) | ((moments["minute"] == 120) & (moments["second"].fillna(0) <= 0))
        moments = moments[mask].copy()

    # --- Sort by match order ---
    sort_cols = [c for c in ["period", "minute", "second", "timestamp"] if c in moments.columns]
    if sort_cols:
        moments = moments.sort_values(sort_cols, ascending=True, kind="stable")

    # --- Group by possession: keep last in each possession (if possession exists) ---
    if "possession" in moments.columns:
        # Secondary sort to ensure "last()" is deterministic with timestamp when available
        sort_cols2 = [c for c in ["possession", "period", "minute", "second", "timestamp"] if c in moments.columns]
        moments = moments.sort_values(sort_cols2, ascending=True, kind="stable")
        moments = moments.groupby("possession", as_index=False).last()

    # --- Drop exact duplicates if any ---
    moments = moments.drop_duplicates()

    # --- Create a tidy projection with stable columns ---
    def safe(val):
        if val is None:
            return None
        if isinstance(val, float) and (math.isnan(val) or math.isinf(val)):
            return None
        return val

    # Build a clean DataFrame with key fields; keep event `id`
    keep_cols = []
    for c in ["id","match_id","type","team","player","minute","second","timestamp","period","possession","xg","shot_outcome"]:
        if c in moments.columns:
            keep_cols.append(c)

    moments_df = moments[keep_cols].copy()

    # Label "Penalty Won" outcome for the Foul Won + flag case (nice-to-have)
    if {"type", "shot_outcome", "foul_won_penalty"}.issubset(moments.columns):
        mask_pw = (moments["type"] == "Foul Won") & (moments["foul_won_penalty"] == True)
        moments_df.loc[mask_pw.values, "shot_outcome"] = "Penalty Won"

    # JSON-safe list of dicts
    def json_fallback(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        return str(o)

    # Replace NaNs with None for JSON
    moments_df = moments_df.replace([np.nan, np.inf, -np.inf], None)

    moments_list: List[Dict[str, Any]] = []
    for _, row in moments_df.iterrows():
        # Compact "who" label like "C. Immobile"
        who = "N/A"
        player = row.get("player", None)
        if isinstance(player, str) and player.strip():
            parts = player.split()
            who = f"{parts[0][0]}. {parts[-1]}" if len(parts) >= 2 else parts[0]

        # "when" as mm:ss if minute/second present
        m, s = row.get("minute", None), row.get("second", None)
        when = f"{int(m)}:{int(s):02d}" if (m is not None and s is not None) else row.get("timestamp", None) or "N/A"

        item = {
            "event_id": row.get("id"),
            "match_id": row.get("match_id", None),
            "possession": row.get("possession", None),
            "who": who,
            "when": when,
            "event_type": row.get("type", None),
            "xg": row.get("xg", None),
            "outcome": row.get("shot_outcome", None),
            "team": row.get("team", None),
        }
        # JSON-safe coercion
        for k, v in list(item.items()):
            try:
                item[k] = v if v is None else json.loads(json.dumps(v, default=json_fallback))
            except Exception:
                item[k] = None if v is None else str(v)
        moments_list.append(item)

    return moments_df, moments_list

In [22]:
# %% Main Execution
FILENAME = "../events_by_match/events_3788741.csv"

# Step 1: Extract important moments
print("Extracting important moments...")
moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

print(f"\nFound {len(moments_df)} important moments")
print(f"  - Shots: {len(moments_df[moments_df['type'] == 'Shot'])}")
print(f"  - Own Goals: {len(moments_df[moments_df['type'] == 'Own Goal Against'])}")
print(f"  - Penalties Won: {len(moments_df[moments_df.get('shot_outcome', '') == 'Penalty Won'])}")

# Step 2: Classify shots with Case 1
print("\n\nClassifying shots...")
shot_moments_labeled = classify_shot_moments(
    FILENAME, 
    moments_df, 
    long_shot_yards=22.0,
    case1_threshold_meters=5.0
)

print(f"\nClassified {len(shot_moments_labeled)} shots")

# Summary of labels
print("\nLabel distribution:")
label_counts = shot_moments_labeled['labels_str'].value_counts()
print(label_counts)

# Show Case 1 shots
case1_shots = shot_moments_labeled[shot_moments_labeled['labels_str'] == 'case1_pass_through_ball']
print(f"\nShots with Case 1 label: {len(case1_shots)}")
if not case1_shots.empty:
    print("\nCase 1 shots:")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(case1_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Display all results
print("\n\nAll labeled shots:")
with pd.option_context("display.max_columns", 50, "display.width", 180):
    print(shot_moments_labeled.to_string(index=False))

Extracting important moments...

Found 7 important moments
  - Shots: 6
  - Own Goals: 1
  - Penalties Won: 0


Classifying shots...

Classified 6 shots

Label distribution:
labels_str
case1_pass_through_ball    5
other                      1
Name: count, dtype: int64

Shots with Case 1 label: 5

Case 1 shots:
                                  id           player  minute  second       xg              labels_str
74750b41-c816-4a6c-88d3-b98859039c74  Lorenzo Insigne      17       3 0.098552 case1_pass_through_ball
c07133d1-fa88-45f8-8a40-7be381ddffd1 Domenico Berardi      44      33 0.128881 case1_pass_through_ball
c12c958c-b795-4471-b81c-c16582d8205a  Lorenzo Insigne      56      21 0.086273 case1_pass_through_ball
ead96180-5c0c-4ccb-b926-d215ba883fcb  Lorenzo Insigne      78      30 0.191777 case1_pass_through_ball
7d0764cb-f0d1-4424-9c97-e83a01388805     Burak Yılmaz      91      55 0.114827 case1_pass_through_ball


All labeled shots:
                                  id  match_id   

In [36]:
# %% Inspect pass flags (pass_* columns with value 1) for shots from important moments
FILENAME = "../events_by_match/events_3788741.csv"

# Load events and important moments
ev = pd.read_csv(FILENAME)
moments_df, _ = important_moments_from_file(FILENAME, xg_threshold=0.08)

# Get shots from important moments
shot_moments = moments_df[moments_df["type"] == "Shot"].copy()

# Merge with full events to get shot_key_pass_id
shot_moments_with_data = shot_moments.merge(
    ev[["id", "shot_key_pass_id"]], 
    on="id", 
    how="left"
)

# Filter to shots that have a key pass
shots_with_kp = shot_moments_with_data[shot_moments_with_data["shot_key_pass_id"].notna()].copy()

print(f"Total important shots: {len(shot_moments)}")
print(f"Shots with key pass: {len(shots_with_kp)}")
print("\n" + "="*100)

# For each shot with key pass, inspect the pass flags
for idx, shot_row in shots_with_kp.iterrows():
    shot_id = shot_row.get("id")
    kp_id = shot_row.get("shot_key_pass_id", np.nan)
    
    if pd.isna(kp_id):
        continue
    
    # Get the pass row
    pass_row = get_pass_row_by_id(ev, kp_id)
    if pass_row is None:
        continue
    
    print(f"\nShot ID: {shot_id}")
    print(f"  Player: {shot_row.get('player')}")
    print(f"  Minute: {shot_row.get('minute')}:{shot_row.get('second')}")
    print(f"  xG: {shot_row.get('xg')}")
    print(f"  Key Pass ID: {kp_id}")
    
    # Find all columns starting with "pass_" that have value 1 (True)
    pass_flags = {}
    for col in pass_row.index:
        if col.startswith("pass_"):
            val = pass_row[col]
            # Check if value is 1, True, or truthy
            if pd.notna(val) and (val == 1 or val == True or (isinstance(val, (int, float)) and val == 1.0)):
                pass_flags[col] = val
    
    if pass_flags:
        print(f"  Pass flags with value 1:")
        for flag, val in sorted(pass_flags.items()):
            print(f"    - {flag}: {val}")
    else:
        print(f"  No pass_* columns with value 1")
    
    # Show location info and distances
    pass_loc = parse_xy(pass_row.get("location"))
    pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
    if pass_loc and pass_end_loc:
        print(f"  Pass location: {pass_loc} -> {pass_end_loc}")
        
        # Calculate actual pass distance (Pythagorean theorem)
        actual_pass_distance = math.hypot(
            pass_end_loc[0] - pass_loc[0],
            pass_end_loc[1] - pass_loc[1]
        )
        print(f"  Actual pass distance: {actual_pass_distance:.2f}")
        
        # Calculate distance toward goal
        dist_toward = pass_distance_toward_goal(pass_loc, pass_end_loc)
        print(f"  Distance toward goal: {dist_toward:.2f}")

print("\n" + "="*100)
print("\nSummary: Pass flags distribution")
print("-" * 100)

# Collect all pass flags across all shots
all_pass_flags = {}
for idx, shot_row in shots_with_kp.iterrows():
    kp_id = shot_row.get("shot_key_pass_id", np.nan)
    if pd.isna(kp_id):
        continue
    
    pass_row = get_pass_row_by_id(ev, kp_id)
    if pass_row is None:
        continue
    
    for col in pass_row.index:
        if col.startswith("pass_"):
            val = pass_row[col]
            if pd.notna(val) and (val == 1 or val == True or (isinstance(val, (int, float)) and val == 1.0)):
                if col not in all_pass_flags:
                    all_pass_flags[col] = 0
                all_pass_flags[col] += 1

if all_pass_flags:
    print("\nPass flags with value 1 (count across all shots):")
    for flag, count in sorted(all_pass_flags.items(), key=lambda x: x[1], reverse=True):
        print(f"  {flag}: {count} shots")
else:
    print("No pass_* columns with value 1 found in any key passes")

Total important shots: 6
Shots with key pass: 5


Shot ID: 74750b41-c816-4a6c-88d3-b98859039c74
  Player: Lorenzo Insigne
  Minute: 17:3
  xG: 0.0985515
  Key Pass ID: 0f6b342d-4857-44e2-bad2-d967e4f25b5b
  Pass flags with value 1:
    - pass_shot_assist: 1.0
  Pass location: (96.1, 31.6) -> (108.7, 25.2)
  Actual pass distance: 14.13
  Distance toward goal: 6.71

Shot ID: c07133d1-fa88-45f8-8a40-7be381ddffd1
  Player: Domenico Berardi
  Minute: 44:33
  xG: 0.12888117
  Key Pass ID: d646d6a0-b62c-4040-bf93-88ccde45fe95
  Pass flags with value 1:
    - pass_shot_assist: 1.0
    - pass_through_ball: 1.0
  Pass location: (90.7, 23.5) -> (110.1, 27.3)
  Actual pass distance: 19.77
  Distance toward goal: 17.52

Shot ID: c12c958c-b795-4471-b81c-c16582d8205a
  Player: Lorenzo Insigne
  Minute: 56:21
  xG: 0.08627259
  Key Pass ID: fb750b03-9283-463f-b029-98a2c68939a2
  Pass flags with value 1:
    - pass_shot_assist: 1.0
  Pass location: (97.7, 32.5) -> (106.8, 27.0)
  Actual pass distance: 

# Case 5: Layoff & Case 4: Cutback

In [37]:
# %% Helper: Check if pass travels away from goal
def pass_distance_away_from_goal(pass_location: Tuple[float, float], 
                                   pass_end_location: Tuple[float, float]) -> float:
    """
    Calculate how far a pass travels away from the goal.
    
    The goal is fixed at (120, 40) - the right side of the pitch.
    We calculate distance to this goal from both locations and see if the pass
    moves further away from the goal.
    
    Args:
        pass_location: Starting location (x, y)
        pass_end_location: Ending location (x, y)
    
    Returns:
        Distance traveled away from goal (positive if moving away, 0 if not)
    """
    x1, y1 = pass_location
    x2, y2 = pass_end_location
    
    # Distance to goal (120, 40) from start location
    dist_start_to_goal = math.hypot(x1 - 120.0, y1 - 40.0)
    
    # Distance to goal (120, 40) from end location
    dist_end_to_goal = math.hypot(x2 - 120.0, y2 - 40.0)
    
    # Distance traveled away from goal (positive if end is further than start)
    distance_away_from_goal = dist_end_to_goal - dist_start_to_goal
    
    # Return positive value only if moving away from goal, otherwise 0
    return max(0.0, distance_away_from_goal)

In [39]:
# %% Check Case 5: Layoff - pass_cross=0, travels away from goal, distance < 5m
def check_case5_layoff(events_df: pd.DataFrame, 
                        pass_id: Any, 
                        threshold_meters: float = 5.0) -> bool:
    """
    Check if a pass satisfies Case 5 (Layoff):
    - pass_cross column = 0 (False/0/NaN)
    - Pass travels away from goal
    - Pass travels less than threshold_meters
    
    Args:
        events_df: Full events DataFrame
        pass_id: ID of the pass event to check
        threshold_meters: Distance threshold (default 5.0 meters)
    
    Returns:
        True if all conditions are met, False otherwise
    """
    # Find the pass row
    pass_row = get_pass_row_by_id(events_df, pass_id)
    if pass_row is None:
        return False
    
    # Check pass_cross = 0 (False, 0, or NaN)
    pass_cross_val = pass_row.get("pass_cross", np.nan)
    is_not_cross = (
        pd.isna(pass_cross_val) or 
        pass_cross_val == False or 
        pass_cross_val == 0 or
        (isinstance(pass_cross_val, str) and pass_cross_val.lower() in ['false', '0', ''])
    )
    
    if not is_not_cross:
        return False  # pass_cross is not 0, so Case 5 fails
    
    # Get pass locations
    pass_loc = parse_xy(pass_row.get("location"))
    pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
    
    if pass_loc is None or pass_end_loc is None:
        return False  # Missing location data
    
    # Check if pass travels away from goal
    dist_away = pass_distance_away_from_goal(pass_loc, pass_end_loc)
    if dist_away <= 0:
        return False  # Pass is not moving away from goal
    
    # Calculate actual pass distance (Pythagorean theorem)
    actual_pass_distance = math.hypot(
        pass_end_loc[0] - pass_loc[0],
        pass_end_loc[1] - pass_loc[1]
    )
    
    # Check if pass travels less than threshold
    return actual_pass_distance < threshold_meters

In [40]:
# %% Check Case 6: Cutback - pass_cross=0, travels away from goal, starts wider than goal mouth, distance > 5m
def check_case6_cutback(events_df: pd.DataFrame, 
                         pass_id: Any, 
                         threshold_meters: float = 5.0) -> bool:
    """
    Check if a pass satisfies Case 6 (Cutback):
    - pass_cross column = 0 (False/0/NaN)
    - Pass travels away from goal
    - Pass starts wider than goal mouth (Y > 44 OR Y < 36)
    - Pass travels more than threshold_meters
    
    Goal mouth width: Y between 36 and 44 (center at 40, ±4)
    Wider than goal mouth: Y > 44 OR Y < 36
    
    Args:
        events_df: Full events DataFrame
        pass_id: ID of the pass event to check
        threshold_meters: Distance threshold (default 5.0 meters)
    
    Returns:
        True if all conditions are met, False otherwise
    """
    # Find the pass row
    pass_row = get_pass_row_by_id(events_df, pass_id)
    if pass_row is None:
        return False
    
    # Check pass_cross = 0 (False, 0, or NaN)
    pass_cross_val = pass_row.get("pass_cross", np.nan)
    is_not_cross = (
        pd.isna(pass_cross_val) or 
        pass_cross_val == False or 
        pass_cross_val == 0 or
        (isinstance(pass_cross_val, str) and pass_cross_val.lower() in ['false', '0', ''])
    )
    
    if not is_not_cross:
        return False  # pass_cross is not 0, so Case 6 fails
    
    # Get pass locations
    pass_loc = parse_xy(pass_row.get("location"))
    pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
    
    if pass_loc is None or pass_end_loc is None:
        return False  # Missing location data
    
    # Check if pass starts wider than goal mouth (Y > 44 OR Y < 36)
    y_start = pass_loc[1]
    is_wider_than_goal_mouth = (y_start > 44.0) or (y_start < 36.0)
    
    if not is_wider_than_goal_mouth:
        return False  # Pass doesn't start wider than goal mouth
    
    # Check if pass travels away from goal
    dist_away = pass_distance_away_from_goal(pass_loc, pass_end_loc)
    if dist_away <= 0:
        return False  # Pass is not moving away from goal
    
    # Calculate actual pass distance (Pythagorean theorem)
    actual_pass_distance = math.hypot(
        pass_end_loc[0] - pass_loc[0],
        pass_end_loc[1] - pass_loc[1]
    )
    
    # Check if pass travels more than threshold
    return actual_pass_distance > threshold_meters

In [ ]:
# # %% Classify shot moments with Case 1, Case 5, and Case 6 (multiple labels allowed)
# def classify_shot_moments(
#     filename: str,
#     moments_df: pd.DataFrame,
#     long_shot_yards: float = 22.0,
#     case1_threshold_meters: float = 5.0,
#     case5_threshold_meters: float = 5.0,
#     case6_threshold_meters: float = 5.0
# ) -> pd.DataFrame:
#     """
#     Classify important moments where type == 'Shot' with multi-labels and xG.
    
#     IMPORTANT: Cases 1, 5, and 6 can all apply simultaneously - multiple labels allowed
    
#     Labels (all can apply):
#       - Case 1: 'case1_pass_through_ball' if pass_cross=0 AND pass travels > threshold toward goal
#       - Case 5: 'case5_layoff' if pass_cross=0 AND pass travels away from goal AND distance < threshold
#       - Case 6: 'case6_cutback' if pass_cross=0 AND pass travels away from goal AND starts wider than goal mouth (Y>44 OR Y<36) AND distance > threshold
#       - pass_through_ball, pass_cross, pass_cut_back (from key pass flags)
#       - 'Long Shot' if distance to nearest goal center >= long_shot_yards
#       - 'other' if none of the above matched

#     Returns a DataFrame with one row per shot moment.
#     """
#     # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
#     ev = pd.read_csv(filename)

#     # Tolerant type column
#     type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
#     if type_col is None:
#         raise ValueError("No 'type'/'Type' column in events CSV.")

#     # Shots in the full events
#     ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

#     # Ensure an xg column exists on ev_shots
#     if "shot_statsbomb_xg" in ev_shots.columns:
#         ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
#     else:
#         ev_shots["xg"] = np.nan

#     # Focus only on the events that are marked as important moments AND are shots
#     if "type" not in moments_df.columns:
#         raise ValueError("moments_df must include a 'type' column.")
#     shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

#     if not shot_ids:
#         raise ValueError("moments_df must include 'id' for shot events to classify them.")

#     shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

#     # Prepare output rows
#     out_rows: List[Dict[str, Any]] = []

#     for _, r in shots_focus.iterrows():
#         labels: List[str] = []

#         # Check if shot has a key pass
#         kp = r.get("shot_key_pass_id", np.nan)
#         if pd.notna(kp):
#             # Check all cases independently - multiple can apply
#             # Case 1: pass_cross=0 AND travels toward goal > threshold
#             if check_case1_pass_toward_goal(ev, kp, threshold_meters=case1_threshold_meters):
#                 labels.append("case1_pass_through_ball")
            
#             # Case 5: pass_cross=0 AND travels away from goal AND distance < threshold
#             if check_case5_layoff(ev, kp, threshold_meters=case5_threshold_meters):
#                 labels.append("case5_layoff")
            
#             # Case 6: pass_cross=0 AND travels away from goal AND wider than goal mouth AND distance > threshold
#             if check_case6_cutback(ev, kp, threshold_meters=case6_threshold_meters):
#                 labels.append("case6_cutback")
            
#             # Also check original pass flags
#             prow = get_pass_row_by_id(ev, kp)
#             if prow is not None:
#                 if "pass_through_ball" in prow.index and pd.notna(prow["pass_through_ball"]) and bool(prow["pass_through_ball"]):
#                     labels.append("pass_through_ball")
#                 if "pass_cross" in prow.index and pd.notna(prow["pass_cross"]) and bool(prow["pass_cross"]):
#                     labels.append("pass_cross")
#                 if "pass_cut_back" in prow.index and pd.notna(prow["pass_cut_back"]) and bool(prow["pass_cut_back"]):
#                     labels.append("pass_cut_back")

#         # Long shot check
#         loc = parse_xy(r.get("location"))
#         dist_goal = float("nan")
#         if loc is not None:
#             dist_goal = dist_to_nearest_goal(loc)
#             if dist_goal >= float(long_shot_yards):
#                 labels.append("Long Shot")

#         # Other label (only if no other labels)
#         if len(labels) == 0:
#             labels.append("other")

#         # Calculate distance to goal for all shots (for reference)
#         if loc is None:
#             loc = parse_xy(r.get("location"))
#         if loc is not None and pd.isna(dist_goal):
#             dist_goal = dist_to_nearest_goal(loc)

#         out_rows.append({
#             "id": r.get("id"),
#             "match_id": r.get("match_id"),
#             "team": r.get("team"),
#             "player": r.get("player"),
#             "minute": r.get("minute"),
#             "second": r.get("second"),
#             "shot_outcome": r.get("shot_outcome"),
#             "xg": r.get("xg"),
#             "distance_to_goal": dist_goal,
#             "labels": labels,
#             "labels_str": ";".join(labels),
#             "shot_key_pass_id": r.get("shot_key_pass_id"),
#         })

#     shot_moments_labeled = pd.DataFrame(out_rows)

#     # Keep only the original important shot ids order if you want
#     order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
#     if "id" in shot_moments_labeled.columns:
#         shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
#         shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

#     return shot_moments_labeled

In [50]:
# %% Classify shot moments with Case 1, Case 5, and Case 6 ONLY (no other labels)
def classify_shot_moments(
    filename: str,
    moments_df: pd.DataFrame,
    long_shot_yards: float = 22.0,
    case1_threshold_meters: float = 5.0,
    case5_threshold_meters: float = 5.0,
    case6_threshold_meters: float = 5.0
) -> pd.DataFrame:
    """
    Classify important moments where type == 'Shot' with ONLY Cases 1, 5, and 6.
    
    Labels (only these cases):
      - Case 1: 'case1_pass_through_ball' if pass_cross=0 AND pass travels > threshold toward goal
      - Case 5: 'case5_layoff' if pass_cross=0 AND pass travels away from goal AND distance < threshold
      - Case 6: 'case6_cutback' if pass_cross=0 AND pass travels away from goal AND starts wider than goal mouth (Y>44 OR Y<36) AND distance > threshold
      - 'other' if none of the above matched

    Returns a DataFrame with one row per shot moment.
    """
    # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
    ev = pd.read_csv(filename)

    # Tolerant type column
    type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
    if type_col is None:
        raise ValueError("No 'type'/'Type' column in events CSV.")

    # Shots in the full events
    ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

    # Ensure an xg column exists on ev_shots
    if "shot_statsbomb_xg" in ev_shots.columns:
        ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
    else:
        ev_shots["xg"] = np.nan

    # Focus only on the events that are marked as important moments AND are shots
    if "type" not in moments_df.columns:
        raise ValueError("moments_df must include a 'type' column.")
    shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

    if not shot_ids:
        raise ValueError("moments_df must include 'id' for shot events to classify them.")

    shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

    # Prepare output rows
    out_rows: List[Dict[str, Any]] = []

    for _, r in shots_focus.iterrows():
        labels: List[str] = []

        # Check if shot has a key pass
        kp = r.get("shot_key_pass_id", np.nan)
        if pd.notna(kp):
            # Only check Cases 1, 5, and 6
            # Case 1: pass_cross=0 AND travels toward goal > threshold
            if check_case1_pass_toward_goal(ev, kp, threshold_meters=case1_threshold_meters):
                labels.append("case1_pass_through_ball")
            
            # Case 5: pass_cross=0 AND travels away from goal AND distance < threshold
            if check_case5_layoff(ev, kp, threshold_meters=case5_threshold_meters):
                labels.append("case5_layoff")
            
            # Case 6: pass_cross=0 AND travels away from goal AND wider than goal mouth AND distance > threshold
            if check_case6_cutback(ev, kp, threshold_meters=case6_threshold_meters):
                labels.append("case6_cutback")

        # Other label (only if no cases matched)
        if len(labels) == 0:
            labels.append("other")

        # Calculate distance to goal for all shots (for reference)
        loc = parse_xy(r.get("location"))
        dist_goal = float("nan")
        if loc is not None:
            dist_goal = dist_to_nearest_goal(loc)

        out_rows.append({
            "id": r.get("id"),
            "match_id": r.get("match_id"),
            "team": r.get("team"),
            "player": r.get("player"),
            "minute": r.get("minute"),
            "second": r.get("second"),
            "shot_outcome": r.get("shot_outcome"),
            "xg": r.get("xg"),
            "distance_to_goal": dist_goal,
            "labels": labels,
            "labels_str": ";".join(labels),
            "shot_key_pass_id": r.get("shot_key_pass_id"),
        })

    shot_moments_labeled = pd.DataFrame(out_rows)

    # Keep only the original important shot ids order if you want
    order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
    if "id" in shot_moments_labeled.columns:
        shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
        shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

    return shot_moments_labeled

In [53]:
# %% Main Execution with Cases 1, 5, and 6 (multiple labels allowed)
FILENAME = "../events_by_match/events_3788754.csv"

# Step 1: Extract important moments
print("Extracting important moments...")
moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

print(f"\nFound {len(moments_df)} important moments")
print(f"  - Shots: {len(moments_df[moments_df['type'] == 'Shot'])}")
print(f"  - Own Goals: {len(moments_df[moments_df['type'] == 'Own Goal Against'])}")
print(f"  - Penalties Won: {len(moments_df[moments_df.get('shot_outcome', '') == 'Penalty Won'])}")

# Step 2: Classify shots with Cases 1, 5, and 6
print("\n\nClassifying shots...")
shot_moments_labeled = classify_shot_moments(
    FILENAME, 
    moments_df, 
    long_shot_yards=22.0,
    case1_threshold_meters=5.0,
    case5_threshold_meters=5.0,
    case6_threshold_meters=5.0
)

print(f"\nClassified {len(shot_moments_labeled)} shots")

# Summary of labels
print("\nLabel distribution (single labels):")
label_counts = shot_moments_labeled['labels_str'].value_counts()
print(label_counts)

# Show shots with Case 1 label (may have multiple labels)
case1_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case1_pass_through_ball', na=False)]
print(f"\nShots with Case 1 label: {len(case1_shots)}")
if not case1_shots.empty:
    print("\nCase 1 shots (may have multiple labels):")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(case1_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Show shots with Case 5 label (may have multiple labels)
case5_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case5_layoff', na=False)]
print(f"\nShots with Case 5 label: {len(case5_shots)}")
if not case5_shots.empty:
    print("\nCase 5 (Layoff) shots (may have multiple labels):")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(case5_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Show shots with Case 6 label (may have multiple labels)
case6_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case6_cutback', na=False)]
print(f"\nShots with Case 6 label: {len(case6_shots)}")
if not case6_shots.empty:
    print("\nCase 6 (Cutback) shots (may have multiple labels):")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(case6_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Show shots with multiple labels
multi_label_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.count(';') > 0]
print(f"\nShots with multiple labels: {len(multi_label_shots)}")
if not multi_label_shots.empty:
    print("\nMultiple label shots:")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(multi_label_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Display all results
print("\n\nAll labeled shots:")
with pd.option_context("display.max_columns", 50, "display.width", 180):
    print(shot_moments_labeled.to_string(index=False))

Extracting important moments...

Found 8 important moments
  - Shots: 8
  - Own Goals: 0
  - Penalties Won: 0


Classifying shots...

Classified 8 shots

Label distribution (single labels):
labels_str
other                      4
case1_pass_through_ball    2
case6_cutback              2
Name: count, dtype: int64

Shots with Case 1 label: 2

Case 1 shots (may have multiple labels):
                                  id              player  minute  second       xg              labels_str
f89401cc-bed2-493a-b129-8df424a434f8 Leonardo Spinazzola      36      41 0.201227 case1_pass_through_ball
7e1297b8-f312-4814-ab12-2a63947e5152       Ciro Immobile      72      18 0.103825 case1_pass_through_ball

Shots with Case 5 label: 0

Shots with Case 6 label: 2

Case 6 (Cutback) shots (may have multiple labels):
                                  id           player  minute  second       xg    labels_str
7cef7d46-56d0-4973-8272-c7e3e962652d Manuel Locatelli      51       4 0.053492 case6_cutback
0cae

In [ ]:
# %% Print pass details for Case 5 and Case 6 shots
FILENAME = "../events_by_match/events_3788754.csv"

# Load events and important moments
ev = pd.read_csv(FILENAME)
moments_df, _ = important_moments_from_file(FILENAME, xg_threshold=0.08)

# Classify shots
shot_moments_labeled = classify_shot_moments(
    FILENAME,
    moments_df,
    long_shot_yards=22.0,
    case1_threshold_meters=5.0,
    case5_threshold_meters=5.0,
    case6_threshold_meters=5.0
)

# Filter shots with Case 5 or Case 6
case5_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case5_layoff', na=False)]
case6_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case6_cutback', na=False)]

print("="*100)
print("CASE 5 (LAYOFF) SHOTS")
print("="*100)

if case5_shots.empty:
    print("No shots with Case 5 label found.")
else:
    print(f"\nFound {len(case5_shots)} shot(s) with Case 5 label\n")
    
    for idx, shot_row in case5_shots.iterrows():
        shot_id = shot_row.get("id")
        kp_id = shot_row.get("shot_key_pass_id", np.nan)
        
        if pd.isna(kp_id):
            print(f"\nShot ID: {shot_id}")
            print(f"  No key pass ID")
            continue
        
        # Get the pass row
        pass_row = get_pass_row_by_id(ev, kp_id)
        if pass_row is None:
            print(f"\nShot ID: {shot_id}")
            print(f"  Key pass ID {kp_id} not found in events")
            continue
        
        print(f"\nShot ID: {shot_id}")
        print(f"  Player: {shot_row.get('player')}")
        print(f"  Minute: {shot_row.get('minute')}:{shot_row.get('second')}")
        print(f"  xG: {shot_row.get('xg')}")
        print(f"  Labels: {shot_row.get('labels_str')}")
        print(f"  Key Pass ID: {kp_id}")
        
        # Pass details
        print(f"\n  Pass details:")
        print(f"    pass_cross: {pass_row.get('pass_cross')}")
        print(f"    location: {pass_row.get('location')}")
        print(f"    pass_end_location: {pass_row.get('pass_end_location')}")
        
        # Parse locations
        pass_loc = parse_xy(pass_row.get("location"))
        pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
        
        if pass_loc and pass_end_loc:
            print(f"\n    Parsed start location: {pass_loc}")
            print(f"    Parsed end location: {pass_end_loc}")
            
            # Calculate distances
            dist_start_to_goal = math.hypot(pass_loc[0] - 120.0, pass_loc[1] - 40.0)
            dist_end_to_goal = math.hypot(pass_end_loc[0] - 120.0, pass_end_loc[1] - 40.0)
            dist_toward = pass_distance_toward_goal(pass_loc, pass_end_loc)
            dist_away = pass_distance_away_from_goal(pass_loc, pass_end_loc)
            actual_dist = math.hypot(pass_end_loc[0] - pass_loc[0], pass_end_loc[1] - pass_loc[1])
            
            print(f"\n    Distance to goal at start: {dist_start_to_goal:.2f}")
            print(f"    Distance to goal at end: {dist_end_to_goal:.2f}")
            print(f"    Distance traveled toward goal: {dist_toward:.2f}")
            print(f"    Distance traveled away from goal: {dist_away:.2f}")
            print(f"    Actual pass distance: {actual_dist:.2f}")
            print(f"    Pass start Y coordinate: {pass_loc[1]:.2f}")
            
            # Verify Case 5 conditions
            print(f"\n    Case 5 conditions:")
            print(f"      - pass_cross = 0: {pass_row.get('pass_cross') == 0 or pd.isna(pass_row.get('pass_cross'))}")
            print(f"      - Pass travels away from goal: {dist_away > 0}")
            print(f"      - Pass distance < 5.0: {actual_dist < 5.0}")

print("\n" + "="*100)
print("CASE 6 (CUTBACK) SHOTS")
print("="*100)

if case6_shots.empty:
    print("No shots with Case 6 label found.")
else:
    print(f"\nFound {len(case6_shots)} shot(s) with Case 6 label\n")
    
    for idx, shot_row in case6_shots.iterrows():
        shot_id = shot_row.get("id")
        kp_id = shot_row.get("shot_key_pass_id", np.nan)
        
        if pd.isna(kp_id):
            print(f"\nShot ID: {shot_id}")
            print(f"  No key pass ID")
            continue
        
        # Get the pass row
        pass_row = get_pass_row_by_id(ev, kp_id)
        if pass_row is None:
            print(f"\nShot ID: {shot_id}")
            print(f"  Key pass ID {kp_id} not found in events")
            continue
        
        print(f"\nShot ID: {shot_id}")
        print(f"  Player: {shot_row.get('player')}")
        print(f"  Minute: {shot_row.get('minute')}:{shot_row.get('second')}")
        print(f"  xG: {shot_row.get('xg')}")
        print(f"  Labels: {shot_row.get('labels_str')}")
        print(f"  Key Pass ID: {kp_id}")
        
        # Pass details
        print(f"\n  Pass details:")
        print(f"    pass_cross: {pass_row.get('pass_cross')}")
        print(f"    location: {pass_row.get('location')}")
        print(f"    pass_end_location: {pass_row.get('pass_end_location')}")
        
        # Parse locations
        pass_loc = parse_xy(pass_row.get("location"))
        pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
        
        if pass_loc and pass_end_loc:
            print(f"\n    Parsed start location: {pass_loc}")
            print(f"    Parsed end location: {pass_end_loc}")
            
            # Calculate distances
            dist_start_to_goal = math.hypot(pass_loc[0] - 120.0, pass_loc[1] - 40.0)
            dist_end_to_goal = math.hypot(pass_end_loc[0] - 120.0, pass_end_loc[1] - 40.0)
            dist_toward = pass_distance_toward_goal(pass_loc, pass_end_loc)
            dist_away = pass_distance_away_from_goal(pass_loc, pass_end_loc)
            actual_dist = math.hypot(pass_end_loc[0] - pass_loc[0], pass_end_loc[1] - pass_loc[1])
            
            print(f"\n    Distance to goal at start: {dist_start_to_goal:.2f}")
            print(f"    Distance to goal at end: {dist_end_to_goal:.2f}")
            print(f"    Distance traveled toward goal: {dist_toward:.2f}")
            print(f"    Distance traveled away from goal: {dist_away:.2f}")
            print(f"    Actual pass distance: {actual_dist:.2f}")
            print(f"    Pass start Y coordinate: {pass_loc[1]:.2f}")
            print(f"    Starts wider than goal mouth (Y>44 OR Y<36): {(pass_loc[1] > 44.0) or (pass_loc[1] < 36.0)}")
            
            # Verify Case 6 conditions
            print(f"\n    Case 6 conditions:")
            print(f"      - pass_cross = 0: {pass_row.get('pass_cross') == 0 or pd.isna(pass_row.get('pass_cross'))}")
            print(f"      - Pass travels away from goal: {dist_away > 0}")
            print(f"      - Starts wider than goal mouth (Y>44 OR Y<36): {(pass_loc[1] > 44.0) or (pass_loc[1] < 36.0)}")
            print(f"      - Pass distance > 5.0: {actual_dist > 5.0}")

print("\n" + "="*100)

CASE 5 (LAYOFF) SHOTS
No shots with Case 5 label found.

CASE 6 (CUTBACK) SHOTS

Found 2 shot(s) with Case 6 label


Shot ID: 7cef7d46-56d0-4973-8272-c7e3e962652d
  Player: Manuel Locatelli
  Minute: 51:4
  xG: 0.053491957
  Labels: case6_cutback
  Key Pass ID: 4fddd778-e21a-4348-819f-b91a100aff50

  Pass details:
    pass_cross: nan
    location: [96.3, 50.2]
    pass_end_location: [93.5, 33.5]

    Parsed start location: (96.3, 50.2)
    Parsed end location: (93.5, 33.5)

    Distance to goal at start: 25.80
    Distance to goal at end: 27.29
    Distance traveled toward goal: 0.00
    Distance traveled away from goal: 1.48
    Actual pass distance: 16.93
    Pass start Y coordinate: 50.20
    Starts wider than goal mouth (Y>44 OR Y<36): True

    Case 6 conditions:
      - pass_cross = 0: True
      - Pass travels away from goal: True
      - Starts wider than goal mouth (Y>44 OR Y<36): True
      - Pass distance > 5.0: True

Shot ID: 0cae89b6-2fe0-4454-b6e2-7d3becdb5fb4
  Player: S

In [52]:
# %% Find files with Case 5 or Case 6 labels
import os
import glob

# Path to events folder
EVENTS_FOLDER = "../events_by_match"

# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(EVENTS_FOLDER, "events_*.csv"))
csv_files.sort()

print(f"Found {len(csv_files)} event files to process")
print("="*80)

files_with_case5_or_case6 = []

for csv_file in csv_files:
    try:
        # Extract important moments
        moments_df, _ = important_moments_from_file(csv_file, xg_threshold=0.08)
        
        # Get shots from important moments
        shot_moments = moments_df[moments_df["type"] == "Shot"]
        
        if shot_moments.empty:
            continue
        
        # Classify shots
        shot_moments_labeled = classify_shot_moments(
            csv_file,
            moments_df,
            long_shot_yards=22.0,
            case1_threshold_meters=5.0,
            case5_threshold_meters=5.0,
            case6_threshold_meters=5.0
        )
        
        # Check for Case 5 or Case 6
        has_case5 = shot_moments_labeled['labels_str'].str.contains('case5_layoff', na=False).any()
        has_case6 = shot_moments_labeled['labels_str'].str.contains('case6_cutback', na=False).any()
        
        if has_case5 or has_case6:
            filename = os.path.basename(csv_file)
            case5_count = shot_moments_labeled['labels_str'].str.contains('case5_layoff', na=False).sum()
            case6_count = shot_moments_labeled['labels_str'].str.contains('case6_cutback', na=False).sum()
            
            files_with_case5_or_case6.append({
                'file': filename,
                'case5_count': case5_count,
                'case6_count': case6_count
            })
            
            print(f"✓ {filename}")
            if has_case5:
                print(f"  - Case 5 (Layoff): {case5_count} shot(s)")
            if has_case6:
                print(f"  - Case 6 (Cutback): {case6_count} shot(s)")
            
            # Stop after finding at least 2 files
            if len(files_with_case5_or_case6) >= 6:
                break
    
    except Exception as e:
        print(f"✗ Error processing {os.path.basename(csv_file)}: {str(e)}")
        continue

print("\n" + "="*80)
print(f"\nSummary: Found {len(files_with_case5_or_case6)} file(s) with Case 5 or Case 6:")
for item in files_with_case5_or_case6:
    print(f"  - {item['file']} (Case 5: {item['case5_count']}, Case 6: {item['case6_count']})")

Found 51 event files to process
✓ events_3788746.csv
  - Case 5 (Layoff): 1 shot(s)
✓ events_3788749.csv
  - Case 6 (Cutback): 1 shot(s)
✓ events_3788751.csv
  - Case 5 (Layoff): 1 shot(s)
✓ events_3788753.csv
  - Case 6 (Cutback): 1 shot(s)
✓ events_3788754.csv
  - Case 6 (Cutback): 2 shot(s)
✓ events_3788756.csv
  - Case 6 (Cutback): 1 shot(s)


Summary: Found 6 file(s) with Case 5 or Case 6:
  - events_3788746.csv (Case 5: 1, Case 6: 0)
  - events_3788749.csv (Case 5: 0, Case 6: 1)
  - events_3788751.csv (Case 5: 1, Case 6: 0)
  - events_3788753.csv (Case 5: 0, Case 6: 1)
  - events_3788754.csv (Case 5: 0, Case 6: 2)
  - events_3788756.csv (Case 5: 0, Case 6: 1)


# Case 2: early cross & Case 3: cross

In [55]:
# %% Helper: Get pitch zone from coordinates (1-18)
def get_pitch_zone(x: float, y: float) -> int:
    """
    Map (x, y) coordinates to pitch zone (1-18).
    
    Pitch dimensions: 120 x 80 units
    - X-axis: 6 vertical strips (0-20, 20-40, 40-60, 60-80, 80-100, 100-120)
    - Y-axis: 3 horizontal rows (0-26.67, 26.67-53.33, 53.33-80)
    
    Zone numbering:
    Top row:     1,  4,  7, 10, 13, 16
    Middle row:  2,  5,  8, 11, 14, 17
    Bottom row:  3,  6,  9, 12, 15, 18
    
    Args:
        x: X coordinate (0-120)
        y: Y coordinate (0-80)
    
    Returns:
        Zone number (1-18)
    """
    # Vertical strips (X-axis)
    if x < 20:
        col = 0
    elif x < 40:
        col = 1
    elif x < 60:
        col = 2
    elif x < 80:
        col = 3
    elif x < 100:
        col = 4
    else:  # x >= 100 and x <= 120
        col = 5
    
    # Horizontal rows (Y-axis)
    if y < (80/3):  # y < 26.67
        row = 2  # Bottom row
    elif y < (80*2/3):  # y < 53.33
        row = 1  # Middle row
    else:  # y >= 53.33
        row = 0  # Top row
    
    # Zone number formula: row * 6 + col + 1
    zone = row * 6 + col + 1
    return zone

In [56]:
# %% Helper: Load 360 data for a specific event
def load_360_data_for_event(match_id: int, event_id: str, frames_360_folder_path: str) -> pd.DataFrame:
    """
    Load 360 data for a specific event from the frames_360 CSV file.
    
    Args:
        match_id: Match ID (e.g., 3788741)
        event_id: Event ID (pass ID) to filter by
        frames_360_folder_path: Path to frames_360_by_match folder
    
    Returns:
        DataFrame with rows matching the event_id, or empty DataFrame if not found
    """
    # Construct path to 360 data file
    frames_360_file = f"{frames_360_folder_path}/frames_360_{match_id}.csv"
    
    try:
        # Load the CSV file
        df_360 = pd.read_csv(frames_360_file)
        
        # Filter rows where id matches event_id
        filtered = df_360[df_360["id"] == event_id].copy()
        
        return filtered
    except FileNotFoundError:
        print(f"Warning: 360 data file not found: {frames_360_file}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Warning: Error loading 360 data: {e}")
        return pd.DataFrame()

In [57]:
# %% Helper: Count teammates in penalty box using 360 data
def count_teammates_in_box(match_id: int, event_id: str, frames_360_folder_path: str) -> int:
    """
    Count number of teammates in the penalty box at the time of the event.
    
    Penalty box coordinates (approximate):
    - X: [102, 120] (from ~18 yards back to goal line)
    - Y: [18, 62] (centered on Y=40, ±22 units)
    
    Args:
        match_id: Match ID
        event_id: Event ID (pass ID) to get 360 data for
        frames_360_folder_path: Path to frames_360_by_match folder
    
    Returns:
        Number of teammates in the penalty box
    """
    # Load 360 data for this event
    df_360 = load_360_data_for_event(match_id, event_id, frames_360_folder_path)
    
    if df_360.empty:
        return 0
    
    # Penalty box boundaries
    x_min_box = 102
    x_max_box = 120
    y_min_box = 18
    y_max_box = 62
    
    teammate_count = 0
    
    # Iterate through each row (each player/object at the event)
    for _, row in df_360.iterrows():
        # Check if teammate
        is_teammate = row.get("teammate", 0)
        if is_teammate != 1:
            continue
        
        # Parse location
        location_str = row.get("location", None)
        if pd.isna(location_str):
            continue
        
        loc = parse_xy(location_str)
        if loc is None:
            continue
        
        x, y = loc
        
        # Check if location is within penalty box
        if (x_min_box <= x <= x_max_box) and (y_min_box <= y <= y_max_box):
            teammate_count += 1
    
    return teammate_count

In [58]:
# %% Check Case 2: Early Cross - pass_cross=1, zone 10-15, 0 teammates in box
def check_case2_early_cross(
    events_df: pd.DataFrame,
    pass_id: str,
    match_id: int,
    frames_360_folder_path: str
) -> bool:
    """
    Check if a pass satisfies Case 2 (Early Cross):
    - pass_cross column = 1 (True/1)
    - Pass location in zones 10-15
    - 0 teammates in penalty box (using 360 data)
    
    Args:
        events_df: Full events DataFrame
        pass_id: ID of the pass event to check
        match_id: Match ID for 360 data lookup
        frames_360_folder_path: Path to frames_360_by_match folder
    
    Returns:
        True if all conditions are met, False otherwise
    """
    # Find the pass row
    pass_row = get_pass_row_by_id(events_df, pass_id)
    if pass_row is None:
        return False
    
    # Check pass_cross = 1 (True, 1, or truthy)
    pass_cross_val = pass_row.get("pass_cross", np.nan)
    is_cross = (
        pd.notna(pass_cross_val) and 
        (pass_cross_val == 1 or pass_cross_val == True or 
         (isinstance(pass_cross_val, (int, float)) and pass_cross_val == 1.0))
    )
    
    if not is_cross:
        return False  # pass_cross is not 1, so Case 2 fails
    
    # Get pass location
    pass_loc = parse_xy(pass_row.get("location"))
    if pass_loc is None:
        return False  # Missing location data
    
    # Check if pass is in zones 10-15
    zone = get_pitch_zone(pass_loc[0], pass_loc[1])
    if zone not in [10, 11, 12, 13, 14, 15]:
        return False  # Pass not in zones 10-15
    
    # Count teammates in box
    num_teammates = count_teammates_in_box(match_id, pass_id, frames_360_folder_path)
    
    # Check if 0 teammates in box
    return num_teammates == 0

In [59]:
# %% Check Case 3: Cross - pass_cross=1, (zone 10-15 with >=1 teammates) OR (zone 16-18)
def check_case3_cross(
    events_df: pd.DataFrame,
    pass_id: str,
    match_id: int,
    frames_360_folder_path: str
) -> bool:
    """
    Check if a pass satisfies Case 3 (Cross):
    - pass_cross column = 1 (True/1)
    - Either:
      a) Pass location in zones 10-15 AND >=1 teammates in penalty box (using 360 data)
      b) Pass location in zones 16-18 (no teammate check needed)
    
    Args:
        events_df: Full events DataFrame
        pass_id: ID of the pass event to check
        match_id: Match ID for 360 data lookup
        frames_360_folder_path: Path to frames_360_by_match folder
    
    Returns:
        True if all conditions are met, False otherwise
    """
    # Find the pass row
    pass_row = get_pass_row_by_id(events_df, pass_id)
    if pass_row is None:
        return False
    
    # Check pass_cross = 1 (True, 1, or truthy)
    pass_cross_val = pass_row.get("pass_cross", np.nan)
    is_cross = (
        pd.notna(pass_cross_val) and 
        (pass_cross_val == 1 or pass_cross_val == True or 
         (isinstance(pass_cross_val, (int, float)) and pass_cross_val == 1.0))
    )
    
    if not is_cross:
        return False  # pass_cross is not 1, so Case 3 fails
    
    # Get pass location
    pass_loc = parse_xy(pass_row.get("location"))
    if pass_loc is None:
        return False  # Missing location data
    
    # Check zone
    zone = get_pitch_zone(pass_loc[0], pass_loc[1])
    
    # Case 3a: Zones 10-15 with >=1 teammates in box
    if zone in [10, 11, 12, 13, 14, 15]:
        num_teammates = count_teammates_in_box(match_id, pass_id, frames_360_folder_path)
        return num_teammates >= 1
    
    # Case 3b: Zones 16-18 (no teammate check needed)
    if zone in [16, 17, 18]:
        return True
    
    # Not in relevant zones
    return False

In [63]:
# %% Updated classify_shot_moments with Cases 1, 2, 3, 5, and 6 (multiple labels allowed)
def classify_shot_moments(
    filename: str,
    moments_df: pd.DataFrame,
    long_shot_yards: float = 22.0,
    case1_threshold_meters: float = 5.0,
    case5_threshold_meters: float = 5.0,
    case6_threshold_meters: float = 5.0,
    frames_360_folder_path: str = "../frames_360_by_match"
) -> pd.DataFrame:
    """
    Classify important moments where type == 'Shot' with Cases 1, 2, 3, 5, and 6.
    
    IMPORTANT: Multiple labels allowed - all applicable cases can be assigned
    
    Labels (all can apply):
      - Case 1: 'case1_pass_through_ball' if pass_cross=0 AND pass travels > threshold toward goal
      - Case 2: 'case2_early_cross' if pass_cross=1 AND zone 10-15 AND 0 teammates in box
      - Case 3: 'case3_cross' if pass_cross=1 AND ((zone 10-15 AND >=1 teammates) OR zone 16-18)
      - Case 5: 'case5_layoff' if pass_cross=0 AND pass travels away from goal AND distance < threshold
      - Case 6: 'case6_cutback' if pass_cross=0 AND pass travels away from goal AND starts wider than goal mouth AND distance > threshold
      - 'other' if none of the above matched

    Returns a DataFrame with one row per shot moment.
    """
    # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
    ev = pd.read_csv(filename)

    # Tolerant type column
    type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
    if type_col is None:
        raise ValueError("No 'type'/'Type' column in events CSV.")

    # Shots in the full events
    ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

    # Ensure an xg column exists on ev_shots
    if "shot_statsbomb_xg" in ev_shots.columns:
        ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
    else:
        ev_shots["xg"] = np.nan

    # Focus only on the events that are marked as important moments AND are shots
    if "type" not in moments_df.columns:
        raise ValueError("moments_df must include a 'type' column.")
    shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

    if not shot_ids:
        raise ValueError("moments_df must include 'id' for shot events to classify them.")

    shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

    # Prepare output rows
    out_rows: List[Dict[str, Any]] = []

    for _, r in shots_focus.iterrows():
        labels: List[str] = []
        
        # Get match_id for 360 data lookup
        match_id = r.get("match_id", None)
        if pd.isna(match_id):
            match_id = None

        # Check if shot has a key pass
        kp = r.get("shot_key_pass_id", np.nan)
        if pd.notna(kp) and match_id is not None:
            # Check all cases independently - multiple can apply
            # Case 1: pass_cross=0 AND travels toward goal > threshold
            if check_case1_pass_toward_goal(ev, kp, threshold_meters=case1_threshold_meters):
                labels.append("case1_pass_through_ball")
            
            # Case 2: pass_cross=1 AND zone 10-15 AND 0 teammates in box
            if check_case2_early_cross(ev, kp, match_id, frames_360_folder_path):
                labels.append("case2_early_cross")
            
            # Case 3: pass_cross=1 AND ((zone 10-15 AND >=1 teammates) OR zone 16-18)
            if check_case3_cross(ev, kp, match_id, frames_360_folder_path):
                labels.append("case3_cross")
            
            # Case 5: pass_cross=0 AND travels away from goal AND distance < threshold
            if check_case5_layoff(ev, kp, threshold_meters=case5_threshold_meters):
                labels.append("case5_layoff")
            
            # Case 6: pass_cross=0 AND travels away from goal AND wider than goal mouth AND distance > threshold
            if check_case6_cutback(ev, kp, threshold_meters=case6_threshold_meters):
                labels.append("case6_cutback")

        # Other label (only if no cases matched)
        if len(labels) == 0:
            labels.append("other")

        # Calculate distance to goal for all shots (for reference)
        loc = parse_xy(r.get("location"))
        dist_goal = float("nan")
        if loc is not None:
            dist_goal = dist_to_nearest_goal(loc)

        out_rows.append({
            "id": r.get("id"),
            "match_id": r.get("match_id"),
            "team": r.get("team"),
            "player": r.get("player"),
            "minute": r.get("minute"),
            "second": r.get("second"),
            "shot_outcome": r.get("shot_outcome"),
            "xg": r.get("xg"),
            "distance_to_goal": dist_goal,
            "labels": labels,
            "labels_str": ";".join(labels),
            "shot_key_pass_id": r.get("shot_key_pass_id"),
        })

    shot_moments_labeled = pd.DataFrame(out_rows)

    # Keep only the original important shot ids order if you want
    order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
    if "id" in shot_moments_labeled.columns:
        shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
        shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

    return shot_moments_labeled

In [68]:
# %% Scan event files for Case 2 (5 events) and Case 3 (5 events)
import os
import glob

EVENTS_FOLDER = "../events_by_match"
FRAMES_360_PATH = "../frames_360_by_match"

# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(EVENTS_FOLDER, "events_*.csv"))
csv_files.sort()

print(f"Scanning {len(csv_files)} event files for Case 2 and Case 3...")
print("="*100)

events_with_case2 = []
events_with_case3 = []

for csv_file in csv_files:
    try:
        # Extract important moments
        moments_df, _ = important_moments_from_file(csv_file, xg_threshold=0.08)
        
        # Get shots from important moments
        shot_moments = moments_df[moments_df["type"] == "Shot"]
        
        if shot_moments.empty:
            continue
        
        # Classify shots
        shot_moments_labeled = classify_shot_moments(
            csv_file,
            moments_df,
            long_shot_yards=22.0,
            case1_threshold_meters=5.0,
            case5_threshold_meters=5.0,
            case6_threshold_meters=5.0,
            frames_360_folder_path=FRAMES_360_PATH
        )
        
        # Check for Case 2
        if len(events_with_case2) < 5:
            case2_mask = shot_moments_labeled['labels_str'].str.contains('case2_early_cross', na=False)
            case2_shots = shot_moments_labeled[case2_mask]
            
            for _, shot_row in case2_shots.iterrows():
                if len(events_with_case2) >= 5:
                    break
                
                match_id = shot_row.get("match_id")
                shot_id = shot_row.get("id")
                kp_id = shot_row.get("shot_key_pass_id")
                labels = shot_row.get("labels_str", "")
                
                # Only add if it has Case 2 (may have other labels too)
                if "case2_early_cross" in labels:
                    events_with_case2.append({
                        'file': os.path.basename(csv_file),
                        'match_id': match_id,
                        'shot_id': shot_id,
                        'key_pass_id': kp_id,
                        'player': shot_row.get('player'),
                        'minute': shot_row.get('minute'),
                        'second': shot_row.get('second'),
                        'xg': shot_row.get('xg'),
                        'labels': labels
                    })
        
        # Check for Case 3
        if len(events_with_case3) < 5:
            case3_mask = shot_moments_labeled['labels_str'].str.contains('case3_cross', na=False)
            case3_shots = shot_moments_labeled[case3_mask]
            
            for _, shot_row in case3_shots.iterrows():
                if len(events_with_case3) >= 5:
                    break
                
                match_id = shot_row.get("match_id")
                shot_id = shot_row.get("id")
                kp_id = shot_row.get("shot_key_pass_id")
                labels = shot_row.get("labels_str", "")
                
                # Only add if it has Case 3 (may have other labels too)
                if "case3_cross" in labels:
                    events_with_case3.append({
                        'file': os.path.basename(csv_file),
                        'match_id': match_id,
                        'shot_id': shot_id,
                        'key_pass_id': kp_id,
                        'player': shot_row.get('player'),
                        'minute': shot_row.get('minute'),
                        'second': shot_row.get('second'),
                        'xg': shot_row.get('xg'),
                        'labels': labels
                    })
        
        # Stop if we have both
        if len(events_with_case2) >= 5 and len(events_with_case3) >= 5:
            break
    
    except Exception as e:
        print(f"✗ Error processing {os.path.basename(csv_file)}: {str(e)}")
        continue

print("\n" + "="*100)
print("CASE 2 (EARLY CROSS) EVENTS")
print("="*100)
print(f"\nFound {len(events_with_case2)} event(s) with Case 2")

for i, event in enumerate(events_with_case2, 1):
    print(f"\n{i}. File: {event['file']}")
    print(f"   Match ID: {event['match_id']}")
    print(f"   Shot ID: {event['shot_id']}")
    print(f"   Player: {event['player']}")
    print(f"   Time: {event['minute']}:{event['second']}")
    print(f"   xG: {event['xg']}")
    print(f"   Key Pass ID: {event['key_pass_id']}")
    print(f"   Labels: {event['labels']}")

print("\n" + "="*100)
print("CASE 3 (CROSS) EVENTS")
print("="*100)
print(f"\nFound {len(events_with_case3)} event(s) with Case 3")

for i, event in enumerate(events_with_case3, 1):
    print(f"\n{i}. File: {event['file']}")
    print(f"   Match ID: {event['match_id']}")
    print(f"   Shot ID: {event['shot_id']}")
    print(f"   Player: {event['player']}")
    print(f"   Time: {event['minute']}:{event['second']}")
    print(f"   xG: {event['xg']}")
    print(f"   Key Pass ID: {event['key_pass_id']}")
    print(f"   Labels: {event['labels']}")

print("\n" + "="*100)
print(f"\nSummary:")
print(f"  Case 2 events found: {len(events_with_case2)}/5")
print(f"  Case 3 events found: {len(events_with_case3)}/5")

Scanning 51 event files for Case 2 and Case 3...

CASE 2 (EARLY CROSS) EVENTS

Found 0 event(s) with Case 2

CASE 3 (CROSS) EVENTS

Found 5 event(s) with Case 3

1. File: events_3788742.csv
   Match ID: 3788742
   Shot ID: 4e0a9f53-2178-4a45-bd78-4ae93c7cbbe9
   Player: Jonas Older Wind
   Time: 11:22
   xG: 0.13215932
   Key Pass ID: fc8f2e18-47eb-42b0-8644-5f326c89bfb9
   Labels: case3_cross

2. File: events_3788742.csv
   Match ID: 3788742
   Shot ID: da5fbab8-0347-4369-9056-df040c778889
   Player: Joel Pohjanpalo
   Time: 59:2
   xG: 0.2884817
   Key Pass ID: c6a8b8c1-e63a-47df-bff0-608cfbc649d4
   Labels: case3_cross

3. File: events_3788742.csv
   Match ID: 3788742
   Shot ID: 92110c7a-9818-4829-936d-913c2e5b9aad
   Player: Jannik Vestergaard
   Time: 92:12
   xG: 0.12056617
   Key Pass ID: 5aa91c26-1129-45b1-8dbf-1eab973bc3f5
   Labels: case3_cross

4. File: events_3788745.csv
   Match ID: 3788745
   Shot ID: f09c2400-8459-43ca-9b87-f5cef3dc0e97
   Player: Harry Kane
   Time: 60

In [66]:
# %% Main Execution with Cases 1, 2, 3, 5, and 6
FILENAME = "../events_by_match/events_3788742.csv"
FRAMES_360_PATH = "../frames_360_by_match"

# Step 1: Extract important moments
print("Extracting important moments...")
moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

print(f"\nFound {len(moments_df)} important moments")
print(f"  - Shots: {len(moments_df[moments_df['type'] == 'Shot'])}")

# Step 2: Classify shots with all cases
print("\n\nClassifying shots...")
shot_moments_labeled = classify_shot_moments(
    FILENAME, 
    moments_df, 
    long_shot_yards=22.0,
    case1_threshold_meters=5.0,
    case5_threshold_meters=5.0,
    case6_threshold_meters=5.0,
    frames_360_folder_path=FRAMES_360_PATH
)

print(f"\nClassified {len(shot_moments_labeled)} shots")

# Summary of labels
print("\nLabel distribution:")
label_counts = shot_moments_labeled['labels_str'].value_counts()
print(label_counts)

# Show shots with different cases
case1_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case1_pass_through_ball', na=False)]
case2_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case2_early_cross', na=False)]
case3_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case3_cross', na=False)]
case5_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case5_layoff', na=False)]
case6_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case6_cutback', na=False)]

print(f"\nShots by case:")
print(f"  Case 1: {len(case1_shots)}")
print(f"  Case 2: {len(case2_shots)}")
print(f"  Case 3: {len(case3_shots)}")
print(f"  Case 5: {len(case5_shots)}")
print(f"  Case 6: {len(case6_shots)}")

# Show shots with multiple labels
multi_label_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.count(';') > 0]
print(f"\nShots with multiple labels: {len(multi_label_shots)}")
if not multi_label_shots.empty:
    print("\nMultiple label shots:")
    with pd.option_context("display.max_columns", 50, "display.width", 180):
        print(multi_label_shots[['id', 'player', 'minute', 'second', 'xg', 'labels_str']].to_string(index=False))

# Display all results
print("\n\nAll labeled shots:")
with pd.option_context("display.max_columns", 50, "display.width", 180):
    print(shot_moments_labeled.to_string(index=False))

Extracting important moments...

Found 7 important moments
  - Shots: 6


Classifying shots...

Classified 6 shots

Label distribution:
labels_str
case3_cross                3
case1_pass_through_ball    2
other                      1
Name: count, dtype: int64

Shots by case:
  Case 1: 2
  Case 2: 0
  Case 3: 3
  Case 5: 0
  Case 6: 0

Shots with multiple labels: 0


All labeled shots:
                                  id  match_id    team                  player  minute  second shot_outcome       xg  distance_to_goal                    labels              labels_str                     shot_key_pass_id
4e0a9f53-2178-4a45-bd78-4ae93c7cbbe9   3788742 Denmark        Jonas Older Wind      11      22        Off T 0.132159          7.280110             [case3_cross]             case3_cross fc8f2e18-47eb-42b0-8644-5f326c89bfb9
78cb3024-7db8-439b-9682-ae9ed18baa7b   3788742 Denmark   Pierre-Emile Højbjerg      14      44        Saved 0.086957         10.700467 [case1_pass_through_ball] case1_p

In [67]:
# %% Print pass details for Case 2 and Case 3 shots
FILENAME = "../events_by_match/events_3788742.csv"
FRAMES_360_PATH = "../frames_360_by_match"

# Load events and important moments
ev = pd.read_csv(FILENAME)
moments_df, _ = important_moments_from_file(FILENAME, xg_threshold=0.08)

# Classify shots
shot_moments_labeled = classify_shot_moments(
    FILENAME,
    moments_df,
    long_shot_yards=22.0,
    case1_threshold_meters=5.0,
    case5_threshold_meters=5.0,
    case6_threshold_meters=5.0,
    frames_360_folder_path=FRAMES_360_PATH
)

# Filter shots with Case 2 or Case 3
case2_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case2_early_cross', na=False)]
case3_shots = shot_moments_labeled[shot_moments_labeled['labels_str'].str.contains('case3_cross', na=False)]

print("="*100)
print("CASE 2 (EARLY CROSS) SHOTS")
print("="*100)

if case2_shots.empty:
    print("No shots with Case 2 label found.")
else:
    print(f"\nFound {len(case2_shots)} shot(s) with Case 2 label\n")
    
    for idx, shot_row in case2_shots.iterrows():
        shot_id = shot_row.get("id")
        kp_id = shot_row.get("shot_key_pass_id", np.nan)
        match_id = shot_row.get("match_id")
        
        if pd.isna(kp_id):
            print(f"\nShot ID: {shot_id}")
            print(f"  No key pass ID")
            continue
        
        # Get the pass row
        pass_row = get_pass_row_by_id(ev, kp_id)
        if pass_row is None:
            print(f"\nShot ID: {shot_id}")
            print(f"  Key pass ID {kp_id} not found in events")
            continue
        
        print(f"\nShot ID: {shot_id}")
        print(f"  Player: {shot_row.get('player')}")
        print(f"  Minute: {shot_row.get('minute')}:{shot_row.get('second')}")
        print(f"  xG: {shot_row.get('xg')}")
        print(f"  Labels: {shot_row.get('labels_str')}")
        print(f"  Key Pass ID: {kp_id}")
        print(f"  Match ID: {match_id}")
        
        # Pass details
        print(f"\n  Pass details:")
        print(f"    pass_cross: {pass_row.get('pass_cross')}")
        print(f"    location: {pass_row.get('location')}")
        print(f"    pass_end_location: {pass_row.get('pass_end_location')}")
        
        # Parse locations
        pass_loc = parse_xy(pass_row.get("location"))
        pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
        
        if pass_loc:
            zone = get_pitch_zone(pass_loc[0], pass_loc[1])
            print(f"\n    Parsed start location: {pass_loc}")
            print(f"    Zone: {zone}")
            print(f"    In zones 10-15? {zone in [10, 11, 12, 13, 14, 15]}")
            
            # Count teammates in box
            num_teammates = count_teammates_in_box(match_id, kp_id, FRAMES_360_PATH)
            print(f"    Teammates in penalty box: {num_teammates}")
            
            # Verify Case 2 conditions
            print(f"\n    Case 2 conditions:")
            print(f"      - pass_cross = 1: {pass_row.get('pass_cross') == 1}")
            print(f"      - Pass in zones 10-15: {zone in [10, 11, 12, 13, 14, 15]}")
            print(f"      - 0 teammates in box: {num_teammates == 0}")

print("\n" + "="*100)
print("CASE 3 (CROSS) SHOTS")
print("="*100)

if case3_shots.empty:
    print("No shots with Case 3 label found.")
else:
    print(f"\nFound {len(case3_shots)} shot(s) with Case 3 label\n")
    
    for idx, shot_row in case3_shots.iterrows():
        shot_id = shot_row.get("id")
        kp_id = shot_row.get("shot_key_pass_id", np.nan)
        match_id = shot_row.get("match_id")
        
        if pd.isna(kp_id):
            print(f"\nShot ID: {shot_id}")
            print(f"  No key pass ID")
            continue
        
        # Get the pass row
        pass_row = get_pass_row_by_id(ev, kp_id)
        if pass_row is None:
            print(f"\nShot ID: {shot_id}")
            print(f"  Key pass ID {kp_id} not found in events")
            continue
        
        print(f"\nShot ID: {shot_id}")
        print(f"  Player: {shot_row.get('player')}")
        print(f"  Minute: {shot_row.get('minute')}:{shot_row.get('second')}")
        print(f"  xG: {shot_row.get('xg')}")
        print(f"  Labels: {shot_row.get('labels_str')}")
        print(f"  Key Pass ID: {kp_id}")
        print(f"  Match ID: {match_id}")
        
        # Pass details
        print(f"\n  Pass details:")
        print(f"    pass_cross: {pass_row.get('pass_cross')}")
        print(f"    location: {pass_row.get('location')}")
        print(f"    pass_end_location: {pass_row.get('pass_end_location')}")
        
        # Parse locations
        pass_loc = parse_xy(pass_row.get("location"))
        pass_end_loc = parse_xy(pass_row.get("pass_end_location"))
        
        if pass_loc:
            zone = get_pitch_zone(pass_loc[0], pass_loc[1])
            print(f"\n    Parsed start location: {pass_loc}")
            print(f"    Zone: {zone}")
            print(f"    In zones 10-15? {zone in [10, 11, 12, 13, 14, 15]}")
            print(f"    In zones 16-18? {zone in [16, 17, 18]}")
            
            # Count teammates in box
            num_teammates = count_teammates_in_box(match_id, kp_id, FRAMES_360_PATH)
            print(f"    Teammates in penalty box: {num_teammates}")
            
            # Verify Case 3 conditions
            print(f"\n    Case 3 conditions:")
            print(f"      - pass_cross = 1: {pass_row.get('pass_cross') == 1}")
            if zone in [10, 11, 12, 13, 14, 15]:
                print(f"      - Pass in zones 10-15: True")
                print(f"      - >=1 teammates in box: {num_teammates >= 1}")
            elif zone in [16, 17, 18]:
                print(f"      - Pass in zones 16-18: True")
                print(f"      - No teammate check needed for zones 16-18")

print("\n" + "="*100)

CASE 2 (EARLY CROSS) SHOTS
No shots with Case 2 label found.

CASE 3 (CROSS) SHOTS

Found 3 shot(s) with Case 3 label


Shot ID: 4e0a9f53-2178-4a45-bd78-4ae93c7cbbe9
  Player: Jonas Older Wind
  Minute: 11:22
  xG: 0.13215932
  Labels: case3_cross
  Key Pass ID: fc8f2e18-47eb-42b0-8644-5f326c89bfb9
  Match ID: 3788742

  Pass details:
    pass_cross: 1.0
    location: [105.1, 15.4]
    pass_end_location: [111.9, 39.3]

    Parsed start location: (105.1, 15.4)
    Zone: 18
    In zones 10-15? False
    In zones 16-18? True
    Teammates in penalty box: 3

    Case 3 conditions:
      - pass_cross = 1: True
      - Pass in zones 16-18: True
      - No teammate check needed for zones 16-18

Shot ID: da5fbab8-0347-4369-9056-df040c778889
  Player: Joel Pohjanpalo
  Minute: 59:2
  xG: 0.2884817
  Labels: case3_cross
  Key Pass ID: c6a8b8c1-e63a-47df-bff0-608cfbc649d4
  Match ID: 3788742

  Pass details:
    pass_cross: 1.0
    location: [100.6, 11.0]
    pass_end_location: [111.5, 42.4]

   